# Backblaze SMART telemetry — exploration

Working notebook for understanding the raw data **before** trusting any model built on it.

Every cell reads from `data/raw/`, which is git-ignored. **Nothing here runs without real data present** — that is intentional. If a cell errors because the directory is empty, download the daily files from Backblaze and unzip them into `data/raw/`.

Outputs are cleared before committing. A notebook with stale numbers in it is a notebook that lies about the current dataset — clear outputs with `jupyter nbconvert --clear-output --inplace notebooks/exploration.ipynb`.

**Questions this notebook needs to answer before modelling:**

1. What is actually in the directory, and over what date range?
2. Does the schema drift between files? (It does — Backblaze has added and removed SMART columns over the years.)
3. How rare is failure, really? This sets the no-skill baseline every metric is judged against.
4. Is missingness random, or is it a property of certain drive models?
5. Do the trailing-window features actually separate failing drives from healthy ones?

In [ ]:
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make `src` importable when the notebook runs from notebooks/.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_pipeline import (
    LABEL_COLUMN,
    SMART_RAW_COLUMNS,
    PipelineConfig,
    configure_logging,
    discover_csv_files,
    read_daily_file,
    run_pipeline,
)
from src.features import FeatureConfig, build_feature_matrix, build_features

configure_logging(verbose=False)
logging.getLogger("src").setLevel(logging.WARNING)  # quieten per-file chatter

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

RAW_DIR = REPO_ROOT / "data" / "raw"
print(f"repo root: {REPO_ROOT}")
print(f"raw data : {RAW_DIR}  (exists: {RAW_DIR.exists()})")

## 1. What is in the raw directory?

In [ ]:
paths = discover_csv_files(RAW_DIR)
total_mb = sum(p.stat().st_size for p in paths) / 1024**2

print(f"daily files : {len(paths):,}")
print(f"on disk     : {total_mb:,.1f} MiB")
print(f"first file  : {paths[0].name}")
print(f"last file   : {paths[-1].name}")

In [ ]:
# One file, in full: how wide is it, and what does a row look like?
sample_raw = pd.read_csv(paths[0])
print(f"{paths[0].name}: {sample_raw.shape[0]:,} rows x {sample_raw.shape[1]:,} columns")
print(f"smart_* columns: {sum(c.startswith('smart_') for c in sample_raw.columns):,}")
sample_raw.head()

## 2. Does the schema drift between files?

If it does, concatenating files naively produces silent NaN blocks. `read_daily_file` normalises the column set; this cell measures how much normalising it has to do.

In [ ]:
# Sample up to 24 files spread across the range rather than reading everything.
probe = paths[:: max(1, len(paths) // 24)][:24]

schemas = {p.name: set(pd.read_csv(p, nrows=0).columns) for p in probe}
sizes = pd.Series({name: len(cols) for name, cols in schemas.items()})

print("column count per sampled file:")
print(sizes.to_string())
print(f"\ndistinct schemas among sampled files: {len({frozenset(s) for s in schemas.values()})}")

common = set.intersection(*schemas.values())
union = set.union(*schemas.values())
print(f"columns in every sampled file : {len(common):,}")
print(f"columns in at least one       : {len(union):,}")
print(f"\nunstable columns: {sorted(union - common)[:20]}")

In [ ]:
# Which of the SMART attributes we actually model are unstable across files?
availability = pd.DataFrame(
    {
        "column": list(SMART_RAW_COLUMNS),
        "present_in_sampled_files": [
            sum(col in cols for cols in schemas.values()) for col in SMART_RAW_COLUMNS
        ],
    }
).assign(share=lambda d: d["present_in_sampled_files"] / len(schemas))

availability.sort_values("share")

## 3. Load and label a slice

Start with a bounded slice — a few hundred files is plenty to explore, and the full multi-year set will not fit comfortably in pandas. Raise `max_files` once the questions below are answered.

In [ ]:
MAX_FILES = 180  # ~6 months; raise once the exploration below is settled

config = PipelineConfig(input_dir=RAW_DIR, horizon_days=7, max_files=MAX_FILES)
labelled, summary = run_pipeline(config)
summary.print_report()

### Sanity checks on the label

These are assertions, not observations. If any fails, the label is wrong and nothing downstream is worth reading.

In [ ]:
# A positive label must always sit 1..7 days before that drive's failure date.
positives = labelled.loc[labelled[LABEL_COLUMN] == 1]
gap = (positives["failure_date"] - positives["date"]).dt.days
assert gap.between(1, config.horizon_days).all(), "label window violated"

# No observation may survive on or after its drive's failure date.
assert (labelled["days_to_failure"].dropna() > 0).all(), "post-failure row leaked through"

# A drive with no recorded failure must never carry a positive label.
assert labelled.loc[labelled["failure_date"].isna(), LABEL_COLUMN].eq(0).all()

print(f"label checks passed on {len(labelled):,} rows")
print(f"days-to-failure for positives: min={gap.min()}, max={gap.max()}")

## 4. How rare is failure?

This number *is* the no-skill baseline. A PR-AUC only means something relative to it.

In [ ]:
n = len(labelled)
pos = int(labelled[LABEL_COLUMN].sum())

print(f"rows                      : {n:,}")
print(f"positive labels           : {pos:,}")
print(f"positive rate             : {pos / n:.6%}")
print(f"negatives per positive    : {(n - pos) / pos:,.0f}:1" if pos else "no positives")
print(f"drives that failed        : {labelled['failure_date'].notna().groupby(labelled['serial_number']).any().sum():,}")
print(f"drives total              : {labelled['serial_number'].nunique():,}")

In [ ]:
# Failures over time: a spike usually means a bad batch or a model retirement,
# and it changes how a chronological split should be cut.
monthly = (
    labelled.assign(month=labelled["date"].dt.to_period("M"))
    .groupby("month")
    .agg(
        rows=("serial_number", "size"),
        drives=("serial_number", "nunique"),
        positives=(LABEL_COLUMN, "sum"),
    )
    .assign(positive_rate=lambda d: d["positives"] / d["rows"])
)
monthly

In [ ]:
ax = monthly["positive_rate"].plot(
    kind="bar", figsize=(11, 3.5), title="Positive-label rate by month", rot=45
)
ax.set_ylabel("share of drive-days")
ax.set_xlabel("")
ax.figure.tight_layout()

## 5. Which drive models dominate?

Backblaze's fleet is concentrated in a handful of models. If failures cluster in one model, a classifier can learn "is this a Seagate 4TB" instead of "is this drive dying" — worth knowing before reading any feature-importance plot.

In [ ]:
by_model = (
    labelled.groupby("model")
    .agg(
        drive_days=("serial_number", "size"),
        drives=("serial_number", "nunique"),
        positives=(LABEL_COLUMN, "sum"),
    )
    .assign(positive_rate=lambda d: d["positives"] / d["drive_days"])
    .sort_values("drive_days", ascending=False)
)
by_model.head(15)

## 6. Missingness

Missingness here is rarely random — a SMART attribute is absent because a *drive model* does not report it. That makes it informative, which is an argument for XGBoost's native NaN handling over imputing everything to a median.

In [ ]:
smart_present = [c for c in SMART_RAW_COLUMNS if c in labelled.columns]

nan_overall = labelled[smart_present].isna().mean().rename("nan_share_overall")

# If a column's missingness is concentrated in whole drive models, it is
# structural, not noise.
nan_by_model = labelled.groupby("model")[smart_present].apply(lambda g: g.isna().mean())
structural = (nan_by_model.round(3).nunique() == 1).rename("uniform_across_models")

pd.concat([nan_overall, structural], axis=1).sort_values(
    "nan_share_overall", ascending=False
)

## 7. Build the trailing-window features

Same code path the training run uses — no notebook-only feature logic, because a feature that only exists here cannot be served.

In [ ]:
feature_config = FeatureConfig(window_days=30, min_history_days=7)
featured, feature_summary = build_features(labelled, feature_config)
feature_summary.print_report()

matrix = build_feature_matrix(featured, feature_config)
print(f"\nX: {matrix.X.shape}   positives: {matrix.positive_count:,}")

## 8. Do the features separate the classes?

If no feature shifts between the classes, no amount of model tuning will help. Point-biserial correlation is a crude first look — it only sees monotone linear association — but a feature with a visible shift here is a feature the tree models will find.

In [ ]:
y = matrix.y.astype(float)
correlations = (
    matrix.X.corrwith(y)
    .rename("corr_with_label")
    .to_frame()
    .assign(abs_corr=lambda d: d["corr_with_label"].abs())
    .sort_values("abs_corr", ascending=False)
    .drop(columns="abs_corr")
)
correlations.head(20)

In [ ]:
# Class-conditional medians for the attributes Backblaze flags as predictive.
# Medians rather than means: these counters are extremely long-tailed.
watch = [
    c
    for c in [
        "smart_5_raw_max_30d",
        "smart_5_raw_change_30d",
        "smart_187_raw_max_30d",
        "smart_188_raw_max_30d",
        "smart_197_raw_max_30d",
        "smart_197_raw_change_30d",
        "smart_198_raw_max_30d",
        "power_on_days",
    ]
    if c in matrix.X.columns
]

comparison = matrix.X[watch].groupby(matrix.y.values).median().T
comparison.columns = ["healthy (label=0)", "fails within 7d (label=1)"]
comparison

In [ ]:
# Share of drive-days with a non-zero reading, by class. For counters that sit
# at zero for most of a drive's life, "is it non-zero at all" carries more
# signal than the magnitude.
nonzero = pd.DataFrame(
    {
        "healthy": (matrix.X.loc[matrix.y == 0, watch] > 0).mean(),
        "fails_within_7d": (matrix.X.loc[matrix.y == 1, watch] > 0).mean(),
    }
)
nonzero.assign(lift=lambda d: d["fails_within_7d"] / d["healthy"].replace(0, np.nan))

## 9. Where to cut the chronological split

The test fold has to contain enough failures to make recall and precision stable. A split that leaves single-digit positives in the test set produces metrics that swing wildly on one drive.

In [ ]:
dates = featured["date"]
span = dates.max() - dates.min()

for fraction in (0.1, 0.2, 0.3):
    cut = dates.min() + span * (1 - fraction)
    test_mask = dates >= cut
    test_positives = int(featured.loc[test_mask, LABEL_COLUMN].sum())
    print(
        f"test_fraction={fraction:.0%}  cut={cut.date()}  "
        f"train_rows={int((~test_mask).sum()):,}  test_rows={int(test_mask.sum()):,}  "
        f"test_positives={test_positives:,}"
    )

## Findings

_Fill this in from the cells above, with the numbers this run actually produced. Do not carry numbers over from a previous run or a different slice._

- Date range and file count loaded:
- Positive rate (the no-skill baseline every metric is judged against):
- Attributes that are structurally missing for some drive models:
- Features showing the clearest class separation:
- Chosen split date, and how many positives land in the test fold: